In [41]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diabetes-prediction-dataset/diabetes_prediction_dataset.csv


### This notebook serves to process the diabetes prediction dataset and split it into a training, development and testing sets and then applying the relevant normalization techniques to speed up learning and exporting the cleaned and finalized sets in the model building phase. 

In [42]:
# Importing Necessary Libraries
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle 

### Importing Diabetes Prediction Dataset

In [43]:
# Loading dataset
df = pd.read_csv('/kaggle/input/diabetes-prediction-dataset/diabetes_prediction_dataset.csv')
df

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0
99997,Male,66.0,0,0,former,27.83,5.7,155,0
99998,Female,24.0,0,0,never,35.42,4.0,100,0


### Removing the gender "Other" since it's only 18 obsevations and will probably have low signal.

In [44]:
# Removing the "Other" gender since it's only 18 observations 
print (df['gender'].value_counts())
df= df[df['gender'].isin(['Male', 'Female'])]
df.columns

gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64


Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'],
      dtype='object')

### Splitting the dataset into training, development and testing set with a ratio of 80% | 10% | 10% 

In [45]:
# Splitting the data set into train/test 
features = ['gender', 'age', 'hypertension','heart_disease', 'smoking_history','bmi', 'HbA1c_level','blood_glucose_level']
target = ['diabetes']
X = df[features]
Y = df[target]
X_train, X_temp, y_train, y_temp = train_test_split(X, Y, test_size = 0.2, random_state = 42)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, random_state = 42)
X_train, X_dev, X_test, y_train, y_dev, y_test = [df.reset_index(drop=True) for df in [X_train, X_dev, X_test, y_train, y_dev, y_test]] 

#### Smoking history column transformed to contain only 4 categories (smoker, non_smoker, former_smoker, unknown) followed by OneHotEncoding to be absorbed in the Model. 

In [46]:
# Defining the encoding function
def encode_smoking_history(df, ohe=None, fit=True):
    """
    df: DataFrame to encode in this case we'll pass X_train and X_test
    ohe: pre-fitted OneHotEncoder (used for test set) fitted on train set to prevent data leakage (good habit!)
    fit: if True, fit a new encoder; if False, transform using the provided encoder
    """
    df = df.copy()
    # Reduce categories
    df['smoking_history'] = df['smoking_history'].replace({
        'never': 'non_smoker',
        'ever': 'non_smoker',
        'No Info': 'unknown',
        'current': 'smoker',
        'former': 'former_smoker',
        'not current': 'former_smoker'
    })
    
    if fit:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
        ohe_df = ohe.fit_transform(df[['smoking_history']])
    else:
        ohe_df = ohe.transform(df[['smoking_history']])
    
    ohe_df.columns = ohe_df.columns.str.replace('smoking_history_', '', regex=False)
    df_encoded = pd.concat([df.drop(columns=['smoking_history']), ohe_df], axis=1)
    
    return df_encoded, ohe
    
X_train_encoded, ohe = encode_smoking_history(X_train, fit=True)
X_dev_encoded,_ = encode_smoking_history(X_dev, ohe = ohe, fit=False)
X_test_encoded,_ = encode_smoking_history(X_test, ohe= ohe, fit=False)

### Gender column mapping

In [47]:
# Encoding the gender column : 1 for Male and 0 for Female
for df in [X_train_encoded, X_dev_encoded, X_test_encoded]:
    df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

### Range of numerical columns

In [48]:
numerical_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
for col in numerical_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"{col}: min = {min_val}, max = {max_val}, range = {max_val - min_val}")

age: min = 0.08, max = 80.0, range = 79.92
bmi: min = 11.09, max = 95.22, range = 84.13
HbA1c_level: min = 3.5, max = 9.0, range = 5.5
blood_glucose_level: min = 80, max = 300, range = 220


### MinMax Normalization on Numerical variables followed by a sanity check to make sure the fitting and transformation were well executed. 

In [49]:
# Fitting MinMaxScaler on train data only 
Scaler = MinMaxScaler()
X_train_encoded.loc[:, numerical_cols] = Scaler.fit_transform(X_train_encoded.loc [:, numerical_cols])
X_dev_encoded.loc [:, numerical_cols] = Scaler.transform(X_dev_encoded.loc [:, numerical_cols])
X_test_encoded.loc [:, numerical_cols] = Scaler.transform(X_test_encoded.loc [:, numerical_cols])
# Scaler check 
for col in numerical_cols:
    print(f"{col} - Train: min={X_train_encoded[col].min():.2f}, max={X_train_encoded[col].max():.2f}")
    print(f"{col} - Dev: min={X_dev_encoded[col].min():.2f}, max={X_dev_encoded[col].max():.2f}")
    print(f"{col} - Test: min={X_test_encoded[col].min():.2f}, max={X_test_encoded[col].max():.2f}\n")

age - Train: min=0.00, max=1.00
age - Dev: min=0.00, max=1.00
age - Test: min=0.00, max=1.00

bmi - Train: min=0.00, max=1.00
bmi - Dev: min=0.00, max=0.92
bmi - Test: min=0.01, max=0.99

HbA1c_level - Train: min=0.00, max=1.00
HbA1c_level - Dev: min=0.00, max=1.00
HbA1c_level - Test: min=0.00, max=1.00

blood_glucose_level - Train: min=0.00, max=1.00
blood_glucose_level - Dev: min=0.00, max=1.00
blood_glucose_level - Test: min=0.00, max=1.00



/tmp/ipykernel_36/1923948477.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.54545455 0.22727273 0.22727273 ... 0.27272727 0.35454545 0.        ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train_encoded.loc[:, numerical_cols] = Scaler.fit_transform(X_train_encoded.loc [:, numerical_cols])
/tmp/ipykernel_36/1923948477.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.36363636 0.02272727 ... 0.20909091 0.29545455 0.36363636]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_dev_encoded.loc [:, numerical_cols] = Scaler.transform(X_dev_encoded.loc [:, numerical_cols])
/tmp/ipykernel_36/1923948477.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.54545455 0.3409

### Saving processed Training, Development and Testing datasets. 

In [50]:
# Finalizing nomenclature for my prepared datasets
X_train_original = X_train.copy()
X_dev_original = X_dev.copy()
X_test_original = X_test.copy() 
X_train = X_train_encoded.copy()
X_dev = X_dev_encoded.copy()
X_test = X_test_encoded.copy() 

### Exporting Datasets

In [53]:
with open("train_dev_test_data.pkl", "wb") as f:
    pickle.dump((X_train, X_dev, X_test, y_train, y_dev, y_test), f)